# Modelo

In [28]:
import numpy as np
import pandas as pd
import joblib
import os

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import randint, uniform, norm, loguniform

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV

from config import SEED

Elegimos dataset

In [8]:
dataset = 'ciclos_c_r2'

## Espectrogramas

Se suelen usar Mel espectrogramas

Traigo los consjuntos de entrenamiento y testeo. (TODO: buscar una manera más eficiente de guardarlos)

In [9]:
train_data = np.load(f'./dataset/{dataset}/train_melspectrogram.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/{dataset}/test_melspectrogram.npz')
X_test = test_data['X']
y_test = test_data['y']

In [10]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((8289, 23808), (8289,), (1296, 23808), (1296,))

### Random Forest

#### Entrenamiento

In [20]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1, max_features='log2')

param_distributions = {
    'max_depth': randint(8, 15),
    'min_samples_split': randint(5, 10),
    'min_samples_leaf': randint(7, 15)
}

In [21]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='recall',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 1/5] END max_depth=14, min_samples_leaf=10, min_samples_split=9;, score=0.660 total time=   2.7s
[CV 2/5] END max_depth=14, min_samples_leaf=10, min_samples_split=9;, score=0.642 total time=   2.4s
[CV 3/5] END max_depth=14, min_samples_leaf=10, min_samples_split=9;, score=0.590 total time=   3.2s
[CV 4/5] END max_depth=14, min_samples_leaf=10, min_samples_split=9;, score=0.595 total time=   4.5s
[CV 5/5] END max_depth=14, min_samples_leaf=10, min_samples_split=9;, score=0.601 total time=   4.6s
[CV 1/5] END max_depth=14, min_samples_leaf=9, min_samples_split=9;, score=0.658 total time=   4.6s
[CV 2/5] END max_depth=14, min_samples_leaf=9, min_samples_split=9;, score=0.640 total time=   4.5s
[CV 3/5] END max_depth=14, min_samples_leaf=9, min_samples_split=9;, score=0.604 total time=   4.6s
[CV 4/5] END max_depth=14, min_samples_leaf=9, min_samples_split=9;, score=0.608 total time=   4.8s
[CV 5/5] END max_depth=14, min_sa

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....002373F3E9B50>, 'min_samples_leaf': <scipy.stats....002373F4426C0>, 'min_samples_split': <scipy.stats....002373F440D70>}"
,n_iter,50
,scoring,'recall'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [22]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(20)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
10,11,7,5,0.636834
17,11,7,8,0.636834
23,13,7,8,0.634333
9,11,8,9,0.625836
28,11,8,6,0.625836
41,11,8,5,0.625836
30,8,12,9,0.625086
12,11,10,7,0.624334
33,11,10,9,0.624334
44,10,7,7,0.624087


In [23]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = RandomForestClassifier(max_depth=11, min_samples_leaf=7, min_samples_split=8, n_jobs=-1, random_state=SEED)
best_model.fit(X_train, y_train)

Best params: {'max_depth': 11, 'min_samples_leaf': 7, 'min_samples_split': 5}
Best CV score: 0.6368336454431961


,n_estimators,100
,criterion,'gini'
,max_depth,11
,min_samples_split,8
,min_samples_leaf,7
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [24]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.90      0.91      4288
           1       0.90      0.92      0.91      4001

    accuracy                           0.91      8289
   macro avg       0.91      0.91      0.91      8289
weighted avg       0.91      0.91      0.91      8289



#### Evaluación

In [25]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.64      0.66       663
           1       0.64      0.69      0.67       633

    accuracy                           0.66      1296
   macro avg       0.66      0.66      0.66      1296
weighted avg       0.66      0.66      0.66      1296



#### Guardado

In [29]:
os.makedirs(f'./modelos/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos/{dataset}/melspec_rf.pkl')

['./modelos/ciclos_c_r2/melspec_rf.pkl']

## Atributos de Audio

1. Resample y Filtro pasa bajos
2. Feature Extractor
    + MFCC (13)
    + ZCR
    + Short-Time Energy
    + SC
    + Spectral Roll-off
    + BER
    + Spectral Flatness

In [30]:
train_data = np.load(f'./dataset/{dataset}/train_features.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/{dataset}/test_features.npz')
X_test = test_data['X']
y_test = test_data['y']

In [31]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((8289, 19), (8289,), (1296, 19), (1296,))

### Random Forest

#### Entrenamiento

In [32]:
rf = RandomForestClassifier(random_state=42, max_features=None, n_jobs=-1)

param_distributions = {
    'max_depth': randint(8, 15),
    'min_samples_split': randint(10, 25),
    'min_samples_leaf': randint(10, 25)
}

In [33]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='recall',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 1/5] END max_depth=14, min_samples_leaf=13, min_samples_split=22;, score=0.670 total time=   1.9s
[CV 2/5] END max_depth=14, min_samples_leaf=13, min_samples_split=22;, score=0.662 total time=   1.9s
[CV 3/5] END max_depth=14, min_samples_leaf=13, min_samples_split=22;, score=0.649 total time=   1.8s
[CV 4/5] END max_depth=14, min_samples_leaf=13, min_samples_split=22;, score=0.641 total time=   1.8s
[CV 5/5] END max_depth=14, min_samples_leaf=13, min_samples_split=22;, score=0.667 total time=   1.8s
[CV 1/5] END max_depth=14, min_samples_leaf=20, min_samples_split=17;, score=0.664 total time=   1.7s
[CV 2/5] END max_depth=14, min_samples_leaf=20, min_samples_split=17;, score=0.640 total time=   1.8s
[CV 3/5] END max_depth=14, min_samples_leaf=20, min_samples_split=17;, score=0.627 total time=   1.7s
[CV 4/5] END max_depth=14, min_samples_leaf=20, min_samples_split=17;, score=0.627 total time=   1.7s
[CV 5/5] END max_dep

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....002373F2CD0F0>, 'min_samples_leaf': <scipy.stats....002373F49AA90>, 'min_samples_split': <scipy.stats....002373F2CD710>}"
,n_iter,50
,scoring,'recall'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [34]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
26,14,12,23,0.658582
31,14,11,19,0.658581
0,14,13,22,0.658082
18,12,12,16,0.656333
32,11,11,19,0.656330
23,12,11,13,0.656081
28,11,11,23,0.656080
38,14,14,17,0.656080
33,13,13,23,0.655333
42,12,10,18,0.654833


In [35]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 14, 'min_samples_leaf': 12, 'min_samples_split': 23}
Best CV score: 0.6585823970037453


In [36]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.90      0.88      4288
           1       0.89      0.86      0.87      4001

    accuracy                           0.88      8289
   macro avg       0.88      0.88      0.88      8289
weighted avg       0.88      0.88      0.88      8289



#### Evaluación

In [37]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.76      0.79      0.78       663
           1       0.77      0.74      0.76       633

    accuracy                           0.77      1296
   macro avg       0.77      0.77      0.77      1296
weighted avg       0.77      0.77      0.77      1296



#### Guardado

In [38]:
joblib.dump(best_model, f'./modelos/{dataset}/features_rf.pkl')

['./modelos/ciclos_c_r2/features_rf.pkl']